# 🎨 Hybrid Blender 4.0.2 GPU Worker (Google Colab + Oracle Hub + AntiGem MCP)

هذه الصفحة تشغّل **Blender 4.0.2** على كرت الشاشة المجاني في Google Colab (**T4 GPU**) وتربطه تلقائياً بسيرفر **Oracle Hub** ومحادثة **AntiGem MCP**.

### قبل التشغيل (لأول مرة فقط):
1. من القائمة العلوية اختر: **Runtime → Change runtime type → T4 GPU**.
2. من الشريط الجانبي الأيسر افتح أيقونة المفتاح (**Secrets 🔑**) وأضف المتغيرات الثلاثة التالية (وفعّل زر **Notebook access** لكل منها):
   - `BLENDER_ORACLE_HOST`
   - `BLENDER_API_TOKEN`
   - `BLENDER_TUNNEL_KEY`
3. اضغط **Runtime → Run all**! (إذا لم تضف Secrets، سيعمل تلقائياً بوضع احتياطي عبر Cloudflare Quick Tunnel).


In [ ]:
#@title 1️⃣ الإعدادات، المفاتيح السرية، وربط Google Drive { display-mode: "form" }
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = "BlenderHybridMCP" #@param {type:"string"}

import base64
import os
import secrets

DRIVE_ROOT = ""
DRIVE_BACKUP_DIR = ""
BLENDER_CACHE_TAR = ""

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
        DRIVE_BACKUP_DIR = f"{DRIVE_ROOT}/workspace"
        BLENDER_CACHE_TAR = f"{DRIVE_ROOT}/cache/blender-4.0.2-linux-x64.tar.xz"
        os.makedirs(f"{DRIVE_ROOT}/cache", exist_ok=True)
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print(f"✅ Google Drive mounted. Backups & cache at: {DRIVE_ROOT}")
    except Exception as exc:
        print(f"⚠️ Could not mount Google Drive ({exc}); continuing without Drive backup.")
        DRIVE_BACKUP_DIR = ""
        BLENDER_CACHE_TAR = ""

def read_secret(name: str) -> str:
    try:
        from google.colab import userdata
        val = userdata.get(name)
        return (val or "").strip()
    except Exception:
        return os.environ.get(name, "").strip()

BLENDER_ORACLE_HOST = read_secret("BLENDER_ORACLE_HOST")
BLENDER_API_TOKEN = read_secret("BLENDER_API_TOKEN")
BLENDER_TUNNEL_KEY = read_secret("BLENDER_TUNNEL_KEY")

USE_ORACLE_TUNNEL = bool(BLENDER_ORACLE_HOST and BLENDER_TUNNEL_KEY and BLENDER_API_TOKEN)
if not BLENDER_API_TOKEN:
    BLENDER_API_TOKEN = secrets.token_hex(24)
    print("⚠️ BLENDER_API_TOKEN secret not found — generated temporary session token for Cloudflare fallback.")

os.environ["BLENDER_API_TOKEN"] = BLENDER_API_TOKEN
os.environ["BLENDER_DRIVE_BACKUP_DIR"] = DRIVE_BACKUP_DIR

print("Mode:", "🔗 Auto Reverse-SSH Tunnel to Oracle Hub" if USE_ORACLE_TUNNEL else "🌐 Cloudflare Quick Tunnel Fallback")


In [ ]:
#@title 2️⃣ فحص GPU وتثبيت Blender 4.0.2 + تحميل Worker { display-mode: "form" }
import os
import shutil
import subprocess
import time
import urllib.request

print("--- GPU Check ---")
subprocess.run(["nvidia-smi"], check=False)

print("\n--- Installing Xvfb & OpenSSH Client ---")
subprocess.run(
    "apt-get -qq update && apt-get -qq install -y xvfb openssh-client libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1",
    shell=True,
    check=True,
)

BLENDER_DIR = "/content/blender"
BLENDER_BIN = f"{BLENDER_DIR}/blender"
BLENDER_URL = "https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz"
LOCAL_TAR = "/content/blender-4.0.2-linux-x64.tar.xz"

if not os.path.isfile(BLENDER_BIN):
    if BLENDER_CACHE_TAR and os.path.isfile(BLENDER_CACHE_TAR):
        print(f"📦 Restoring Blender 4.0.2 from Google Drive cache: {BLENDER_CACHE_TAR}")
        shutil.copy2(BLENDER_CACHE_TAR, LOCAL_TAR)
    else:
        print("⬇️ Downloading official Blender 4.0.2 (Linux x64)...")
        subprocess.run(["curl", "-fL", "--progress-bar", BLENDER_URL, "-o", LOCAL_TAR], check=True)
        if BLENDER_CACHE_TAR:
            try:
                shutil.copy2(LOCAL_TAR, BLENDER_CACHE_TAR)
                print("💾 Saved Blender archive to Google Drive cache for instant future boots!")
            except Exception as exc:
                print(f"⚠️ Could not save cache to Drive: {exc}")

    os.makedirs(BLENDER_DIR, exist_ok=True)
    print("📂 Extracting Blender 4.0.2...")
    subprocess.run(["tar", "-xf", LOCAL_TAR, "-C", BLENDER_DIR, "--strip-components=1"], check=True)
    try:
        os.remove(LOCAL_TAR)
    except OSError:
        pass

ver = subprocess.check_output([BLENDER_BIN, "--version"], text=True).splitlines()[0]
print("✅ Installed:", ver)

WORKER_URL = f"https://raw.githubusercontent.com/lyla360/render-3d-clips/main/colab/blender_worker.py?t={int(time.time())}"
WORKER_PATH = "/content/blender_worker.py"
print("⬇️ Fetching latest blender_worker.py...")
urllib.request.urlretrieve(WORKER_URL, WORKER_PATH)
print("✅ Worker ready at:", WORKER_PATH)


In [ ]:
#@title 3️⃣ تشغيل Blender 4.0.2 GPU Worker وفتح النفق المباشر (يبقى نشطاً أثناء التصميم)
import base64
import json
import os
import re
import subprocess
import time
import urllib.request

WORKSPACE_DIR = "/content/blender_workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

worker_env = os.environ.copy()
worker_env.update({
    "BLENDER_WORKER_HOST": "127.0.0.1",
    "BLENDER_WORKER_PORT": "9877",
    "BLENDER_BACKEND_NAME": "colab",
    "BLENDER_WORKSPACE_DIR": WORKSPACE_DIR,
    "BLENDER_API_TOKEN": BLENDER_API_TOKEN,
    "BLENDER_DRIVE_BACKUP_DIR": DRIVE_BACKUP_DIR,
})

def start_blender_worker():
    log_f = open("/content/blender_worker.log", "a", encoding="utf-8")
    proc = subprocess.Popen(
        [
            "xvfb-run", "-a", "-s", "-screen 0 1280x720x24",
            "/content/blender/blender", "--background", "--python", "/content/blender_worker.py"
        ],
        env=worker_env,
        stdout=log_f,
        stderr=subprocess.STDOUT,
    )
    return proc

def wait_for_health(timeout=40):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://127.0.0.1:9877/health", timeout=2) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                if data.get("ok"):
                    return data
        except Exception:
            pass
        time.sleep(0.5)
    raise RuntimeError("Blender worker failed to become healthy. Check /content/blender_worker.log")

# Terminate any leftover worker from an earlier cell re-run
subprocess.run(["pkill", "-f", "blender_worker.py"], check=False)
time.sleep(0.5)

worker_proc = start_blender_worker()
health = wait_for_health(45)
print("🚀 Blender GPU Worker is LIVE!")
print(json.dumps(health, indent=2, ensure_ascii=False))

if USE_ORACLE_TUNNEL:
    os.makedirs("/root/.ssh", exist_ok=True)
    os.chmod("/root/.ssh", 0o700)
    key_path = "/root/.ssh/bltunnel_ed25519"
    raw_secret = BLENDER_TUNNEL_KEY.strip()
    if "BEGIN OPENSSH PRIVATE KEY" in raw_secret:
        key_bytes = raw_secret.encode("utf-8") + b"\n"
    else:
        key_bytes = base64.b64decode(raw_secret)
    with open(key_path, "wb") as kf:
        kf.write(key_bytes)
    os.chmod(key_path, 0o600)

    print(f"\n🔗 Connecting Colab GPU -> Oracle Hub ({BLENDER_ORACLE_HOST}:19877)...")
    print("✅ عند نجاح الاتصال سيصبح كرت T4 GPU نشطاً تلقائياً في أداة MCP على هاتفك! (أبقِ هذه الخلية قيد التشغيل)\n")

    try:
        while True:
            if worker_proc.poll() is not None:
                print("⚠️ Blender worker exited; restarting...")
                worker_proc = start_blender_worker()
                wait_for_health(45)

            ssh_cmd = [
                "ssh", "-N",
                "-o", "ExitOnForwardFailure=yes",
                "-o", "BatchMode=yes",
                "-o", "ServerAliveInterval=20",
                "-o", "ServerAliveCountMax=3",
                "-o", "StrictHostKeyChecking=accept-new",
                "-i", key_path,
                "-R", "127.0.0.1:19877:127.0.0.1:9877",
                f"bltunnel@{BLENDER_ORACLE_HOST}",
            ]
            print(f"[{time.strftime('%H:%M:%S')}] 🟢 Reverse SSH Tunnel active (Colab GPU <-> Oracle Hub)...")
            rc = subprocess.call(ssh_cmd)
            print(f"[{time.strftime('%H:%M:%S')}] 🔁 Tunnel disconnected (code={rc}); reconnecting in 3s...")
            time.sleep(3)
    except KeyboardInterrupt:
        print("🛑 Stopped by user.")
    finally:
        worker_proc.terminate()
else:
    cf_bin = "/content/cloudflared"
    if not os.path.isfile(cf_bin):
        subprocess.run(
            ["curl", "-fsSL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", cf_bin],
            check=True,
        )
        os.chmod(cf_bin, 0o755)
    print("\n🌐 Starting Cloudflare Quick Tunnel (fallback mode)...")
    cf_proc = subprocess.Popen(
        [cf_bin, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:9877"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url_printed = False
    try:
        for line in cf_proc.stdout:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m and not url_printed:
                url_printed = True
                print("=" * 60)
                print("🌐 Colab Fallback URL:", m.group(0))
                print("🔑 Session Token:     ", BLENDER_API_TOKEN)
                print("استخدم أداة blender_switch_backend مع هذا الرابط والتوكن للربط المؤقت.")
                print("=" * 60)
            if worker_proc.poll() is not None:
                worker_proc = start_blender_worker()
    except KeyboardInterrupt:
        print("🛑 Stopped by user.")
    finally:
        cf_proc.terminate()
        worker_proc.terminate()
